# 03 - Feature Engineering

**Goal:** Prepare the cleaned dataset for modeling — separate features and target, one-hot encode categorical variables, and create a stratified train/test split.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/telco_churn_clean.csv")

In [2]:
df_ml = df.copy()

### Verify target variable

In [3]:
#verifying dtype of target varibale
df_ml["Churn"].value_counts()


Churn
0    5174
1    1869
Name: count, dtype: int64

In [4]:
df_ml["Churn"].dtype

dtype('int64')

### Separate features and target

**Fix:** `customerID` is a unique identifier (one distinct value per row) and carries no predictive signal — but it was being left in the feature set in the original version of this notebook. Left in, it gets one-hot encoded into ~7,000 dummy columns (one per customer!), which massively bloats the feature space for no benefit. Dropping it here alongside the target.

In [5]:
x = df_ml.drop(["Churn", "customerID"], axis=1)
x.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [6]:
#target variable
y = df_ml["Churn"]

### Identify categorical vs. numerical columns

In [7]:
#identifying categorical and numerical columns

cat_cols = x.select_dtypes(include = ["object"]).columns
num_cols = x.select_dtypes(include = ["int64", "float64"]).columns

cat_cols, num_cols

/tmp/ipykernel_759/1587964735.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = x.select_dtypes(include = ["object"]).columns


(Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
        'PaperlessBilling', 'PaymentMethod'],
       dtype='str'),
 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str'))

### Encode categorical features

Using one-hot encoding (`pd.get_dummies`) with `drop_first=True` to avoid the dummy variable trap.

In [8]:
#encoding categorical features 
x_encoded = pd.get_dummies(x, columns = cat_cols, drop_first=True) #one hot encoding


In [9]:
x_encoded.shape

(7043, 30)

### Train/test split

Using a stratified 80/20 split so the class imbalance in `Churn` is preserved in both sets.

In [10]:
#train test split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x_encoded, y, test_size=0.2, random_state=42, stratify=y
)

In [11]:
#checking split 
print(y_train.value_counts(normalize=True))


Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64


In [12]:
print(y_test.value_counts(normalize=True))

Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


### Save processed splits

These files are intentionally excluded from version control via `.gitignore` since they're derived, reproducible artifacts — re-run this notebook to regenerate them.

In [13]:
#saving the data and splits to be used in modeling file
x_train.to_csv("../data/processed/X_train.csv", index=False)
x_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

### Summary

- Split features (`X`) from target (`Churn`)
- One-hot encoded all categorical columns
- Created an 80/20 stratified train/test split
- Saved `X_train`, `X_test`, `y_train`, `y_test` to `data/processed/`

**Next:** [`04_modeling.ipynb`](./04_modeling.ipynb) — train and compare classification models.